In [2]:
import chess, chess.engine, os, stat
from policy import *
import random
from discrim import *
from file_helper import truncateGame

2026-04-23 19:10:48.244303: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-23 19:10:48.288852: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-23 19:10:50.535392: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/storage/icds/RISE/sw8/anaconda/conda_envs/pytorch/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version 

POLICY V8 (JOINT)


In [3]:
from stockfish import Stockfish
engine_path = r"./stockfish/src/stockfish"
sf = Stockfish(engine_path, parameters={"Threads": 1, "Hash": 256})
sf.set_depth(2)
sf.set_skill_level(2)
sf.get_engine_parameters()

{'Debug Log File': '',
 'Threads': 1,
 'Hash': 256,
 'Ponder': False,
 'MultiPV': 1,
 'Skill Level': 2,
 'Move Overhead': 10,
 'Slow Mover': 100,
 'UCI_Chess960': False,
 'UCI_LimitStrength': False,
 'UCI_Elo': 1350,
 'Contempt': 0,
 'Min Split Depth': 0,
 'Minimum Thinking Time': 20}

In [3]:
games= load_json("./data/Bijay_1549_games.json")
print(len(games))

Loading games: 100%|██████████| 392/392 [00:00<00:00, 588.44it/s]

392


In [5]:
import os

def simulate_games(agent, sf, num_games=400, file_dir="./data", file_postfix="0"):
    os.makedirs(file_dir, exist_ok=True)  # ✅ fix: create dir before writing
    
    games_data = []
    i = 0
    while i < num_games:
        board = chess.Board()
        moves = []
        aborted = False
        
        while not board.is_game_over():
            try:
                if board.turn == chess.WHITE:
                    move = agent.act(board)
                else:
                    sf.set_fen_position(board.fen())
                    best = sf.get_best_move()
                    if best is None:
                        aborted = True
                        break
                    move = chess.Move.from_uci(best)
            except Exception as e:
                print(f"[Game {i+1}] Error: {e}")
                aborted = True
                break
            
            board.push(move)
            moves.append(move.uci())
        
        if aborted:
            continue  # ✅ fix: skip broken games instead of saving them
        
        game_data = {
            "event": "Agent vs Stockfish",
            "round": i + 1,
            "white": f"Mimic Agent of {agent.id}",
            "black": "Stockfish",
            "result": board.result(),
            "moves": " ".join(moves)
        }
        if truncateGame(game_data):
            games_data.append(game_data)
            i += 1

    file_postfix = str(file_postfix).replace(".", "_")
    file_path = f"{file_dir}/{agent.id}_agent_vs_stockfish_{file_postfix}.json"
    with open(file_path, "w") as f:
        json.dump(games_data, f, indent=4)
    print(f"[Saved] {file_path}")  # ✅ confirms write succeeded
    return file_path

In [6]:
def overall_similarity_pipeline(json_A, json_B, player_A, player_B):

    print(f"\n--- FULL PIPELINE: {player_A} vs {player_B} ---\n")

    # ============================================================
    # 🔧 FIX: normalize JSON INSIDE PIPELINE (list → string)
    # ============================================================
    def normalize_json(path):
        with open(path, "r") as f:
            games = json.load(f)

        for g in games:
            if isinstance(g.get("moves"), list):
                g["moves"] = " ".join(g["moves"])

        return games

    # Write temporary cleaned versions (no external preprocessing step)
    import tempfile

    def write_temp(games):
        tmp = tempfile.NamedTemporaryFile(delete=False, mode="w", suffix=".json")
        json.dump(games, tmp)
        tmp.close()
        return tmp.name

    clean_A = write_temp(normalize_json(json_A))
    clean_B = write_temp(normalize_json(json_B))

    # ============================================================
    # ORIGINAL PIPELINE (UNCHANGED LOGIC BELOW)
    # ============================================================
    b_A, m_A, l_A = load_json_game_sequences(clean_A, player_A, 1.0)
    b_B, m_B, l_B = load_json_game_sequences(clean_B, player_B, 0.0)

    min_games = min(len(l_A), len(l_B))
    if min_games == 0:
        print("Not enough usable games.")
        return None

    b_A, m_A, l_A = b_A[:min_games], m_A[:min_games], l_A[:min_games]
    b_B, m_B, l_B = b_B[:min_games], m_B[:min_games], l_B[:min_games]

    raw_boards = b_A + b_B
    raw_moves  = m_A + m_B
    raw_labels = l_A + l_B

    combined = list(zip(raw_boards, raw_moves, raw_labels))
    random.shuffle(combined)
    raw_boards, raw_moves, raw_labels = zip(*combined)

    all_boards = np.array(raw_boards)
    all_moves  = np.array(raw_moves)
    all_labels = np.array(raw_labels)

    all_moves_onehot = tf.one_hot(all_moves, NUM_MOVES)

    model = build_style_classifier()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    )

    model.fit(
        x={"board_seq": all_boards, "move_seq": all_moves_onehot},
        y=all_labels,
        batch_size=32,
        epochs=20,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=1
    )

    similarity = compute_overall_similarity(
        clean_A, clean_B, player_A, player_B, model
    )

    print(f"Overall playstyle similarity: {similarity:.2f}%")
    os.remove(clean_A)
    os.remove(clean_B)
    return similarity

In [6]:
def hyper_tuning(games,player_name,sf, num_games=400,file_dir="./data",player_file_dir="./data"):
    a_values = np.linspace(0, 1, 11)
    
    best_a = None
    best_score = float("-inf")
    agent = Agent(player_name,stockfish_path=r"./stockfish/src/stockfish")

    for a in a_values:
        try:

            agent.train(games,alpha=a)
            
            player_file_path = f"{player_file_dir}/{agent.id}_games.json"
            agent_file_path = simulate_games(
                agent,
                sf,
                num_games,
                file_dir=file_dir,
                file_postfix=f"_a_{a:.2f}"
            )
            
            score = overall_similarity_pipeline(
                player_file_path,
                agent_file_path,
                f"{agent.id}",
                f"Mimic Agent of {agent.id}"
            )

            print(f"a={a:.2f}, score={score:.3f}")

            if score > best_score:
                best_score = score
                best_a = a
        
        finally:
            # 🔥 CRITICAL: prevent Colab crashes
            import gc
            tf.keras.backend.clear_session()
            gc.collect()

    print(f"\nBest a: {best_a:.2f} (score={best_score:.3f})")

In [ ]:
hyper_tuning(games,"Bijay_1549",sf,num_games=400,file_dir="./data400")

/storage/home/jmy5612/model V8/policy.py:123: UserWarning: Note that even though you've set Stockfish to play on a weaker elo or skill level, get_evaluation will still return full strength Stockfish's evaluation of the position.
  info = self.sf.get_evaluation()
/storage/home/jmy5612/model V8/policy.py:270: UserWarning: Note that even though you've set Stockfish to play on a weaker elo or skill level, get_top_moves will still return the top moves of full strength Stockfish.
  top = self.sf.get_top_moves(top_k)


Epoch 1/20
783/783 [==============================] - 302s 382ms/step - loss: 0.0025 - accuracy: 0.0965
Epoch 2/20
783/783 [==============================] - 300s 383ms/step - loss: 0.0019 - accuracy: 0.1102
Epoch 3/20
783/783 [==============================] - 300s 383ms/step - loss: 0.0017 - accuracy: 0.1129
Epoch 4/20
783/783 [==============================] - 300s 383ms/step - loss: 0.0015 - accuracy: 0.1139
Epoch 5/20
783/783 [==============================] - 300s 383ms/step - loss: 0.0013 - accuracy: 0.1138
Epoch 6/20
783/783 [==============================] - 300s 383ms/step - loss: 0.0011 - accuracy: 0.1139
Epoch 7/20
783/783 [==============================] - 300s 383ms/step - loss: 8.9686e-04 - accuracy: 0.1142
Epoch 8/20
783/783 [==============================] - 300s 382ms/step - loss: 7.4194e-04 - accuracy: 0.1136
Epoch 9/20
783/783 [==============================] - 300s 383ms/step - loss: 6.0987e-04 - accuracy: 0.1145
Epoch 10/20
783/783 [==============================]

In [ ]:
def play_sf_vs_sf(sf, num_games=400, file_dir="./data"):
    all_games = []

    for i in range(num_games):
        print(f"Game {i+1}/{num_games}")
        board = chess.Board()
        moves = []
        move_number = 1

        while not board.is_game_over():
            fen = board.fen()
            sf.set_fen_position(fen)
            move = sf.get_best_move()
            if move is None:
                break
            moves.append((board.turn, move_number, board.san(chess.Move.from_uci(move))))
            board.push(chess.Move.from_uci(move))
            if board.turn == chess.WHITE:
                move_number += 1

        pgn_moves = []
        for turn, num, san in moves:
            if turn == chess.WHITE:
                pgn_moves.append(f"{num}. {san}")
            else:
                pgn_moves.append(san)

        result = board.result()
        if result == "*":
            result = "1/2-1/2"

        all_games.append({
            "event": "SF vs SF",
            "white": "sf_white",
            "black": "Stockfish",
            "result": result,
            "moves": " ".join(pgn_moves)
        })

        print(f"Result: {result}")

    file_path = f"{file_dir}/sf_white_games.json"
    os.makedirs(file_dir, exist_ok=True)
    with open(file_path, "w") as f:
        json.dump(all_games, f, indent=4)
    print(f"Saved {num_games} games to {file_path}")

    return file_path

In [ ]:
def get_sf_white_similarity(sf, player_name, file_dir="./data"):
    sf_json = play_sf_vs_sf(sf, file_dir=file_dir)
    player_json = f"{file_dir}/{player_name}_games.json"

    similarity = overall_similarity_pipeline(
        json_A=player_json,
        json_B=sf_json,
        player_A=player_name,
        player_B="sf_white"
    )

    print(f"Stockfish white vs {player_name} similarity: {similarity:.2f}%")
    return similarity

similarity = get_sf_white_similarity(
    sf,
    player_name="Bijay_1549"
)

In [7]:
def evaluate_all_alphas(agent_id, player_file_dir="./data", file_dir="./data"):
    a_values = np.linspace(0, 1, 11)
    
    player_file_path = f"{player_file_dir}/{agent_id}_games.json"
    best_a = None
    best_score = float("-inf")

    for a in a_values:
        file_postfix = f"_a_{a:.2f}".replace(".", "_")
        agent_file_path = f"{file_dir}/{agent_id}_agent_vs_stockfish_{file_postfix}.json"
        
        if not os.path.exists(agent_file_path):
            print(f"a={a:.2f}, file not found: {agent_file_path}")
            continue
        
        try:
            score = overall_similarity_pipeline(
                player_file_path,
                agent_file_path,
                f"{agent_id}",
                f"Mimic Agent of {agent_id}"
            )
            print(f"a={a:.2f}, score={score:.3f}")
            
            if score > best_score:
                best_score = score
                best_a = a
        except Exception as e:
            print(f"a={a:.2f}, error: {e}")

    if best_a is not None:
        print(f"\nBest a: {best_a:.2f} (score={best_score:.3f})")
    else:
        print("\nNo successful evaluations.")
    
    return best_a, best_score
evaluate_all_alphas("Bijay_1549",player_file_dir="./data",file_dir="./data400")


--- FULL PIPELINE: Bijay_1549 vs Mimic Agent of Bijay_1549 ---

Epoch 1/20
20/20 [==============================] - 25s 1s/step - loss: 1.7697 - accuracy: 0.6416 - val_loss: 1.7049 - val_accuracy: 0.6815
Epoch 2/20
20/20 [==============================] - 19s 944ms/step - loss: 1.5988 - accuracy: 0.7680 - val_loss: 1.3786 - val_accuracy: 0.9299
Epoch 3/20
20/20 [==============================] - 19s 940ms/step - loss: 1.2026 - accuracy: 0.9568 - val_loss: 1.0135 - val_accuracy: 0.9809
Epoch 4/20
20/20 [==============================] - 19s 941ms/step - loss: 1.0135 - accuracy: 0.9696 - val_loss: 0.9529 - val_accuracy: 0.9873
Epoch 5/20
20/20 [==============================] - 19s 937ms/step - loss: 0.9408 - accuracy: 0.9792 - val_loss: 0.9191 - val_accuracy: 0.9809
Epoch 6/20
20/20 [==============================] - 19s 939ms/step - loss: 0.8798 - accuracy: 0.9920 - val_loss: 0.8895 - val_accuracy: 0.9809
Epoch 7/20
20/20 [==============================] - 19s 939ms/step - loss: 0.835

(0.8, 2.9004777157284334)